In [ ]:
import time
import gc
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

from quant_linear import QuantLinear

In [ ]:
def get_allocated_memory_mb() -> float:
    """Возвращает чистый объем VRAM (в мегабайтах), занятый весами."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    return torch.cuda.memory_allocated() / (1024 * 1024)

def get_peak_memory_mb() -> float:
    """Возвращает пиковое потребление VRAM (включая KV-cache)."""
    return torch.cuda.max_memory_allocated() / (1024 * 1024)

def reset_memory_stats():
    """Сбрасывает счетчик пиковой памяти."""
    torch.cuda.reset_peak_memory_stats()

In [ ]:
def get_wikitext_inputs(tokenizer, context_length: int = 512):
    """
    Загружает WikiText-2, токенизирует и возвращает батч (размером 1) 
    с заданной длиной контекста.
    """
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    
    # Склеиваем первые несколько десятков блоков текста
    text = "\n\n".join(dataset["text"][:200])
    
    # Токенизируем
    inputs = tokenizer(text, return_tensors="pt")
    
    # Отрезаем ровно столько токенов, сколько запросили для контекста (batch=1)
    inputs["input_ids"] = inputs["input_ids"][:, :context_length]
    inputs["attention_mask"] = inputs["attention_mask"][:, :context_length]
    
    assert inputs["input_ids"].shape[1] == context_length, "Недостаточно текста для заданного контекста"
    return inputs

In [ ]:
def measure_generation_speed(model, inputs, num_tokens: int = 128) -> float:
    """Замеряет скорость генерации"""
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    assert inputs["input_ids"].shape[0] == 1, "Batch size должен быть равен 1"
    
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=5, do_sample=False)
        torch.cuda.synchronize()

    start_time = time.perf_counter()
    with torch.no_grad():
        _ = model.generate(
            **inputs, 
            max_new_tokens=num_tokens, 
            min_new_tokens=num_tokens, 
            do_sample=False,
            use_cache=True 
        )
        torch.cuda.synchronize()
    end_time = time.perf_counter()

    time_taken = end_time - start_time
    tokens_per_sec = num_tokens / time_taken
    
    return tokens_per_sec

In [ ]:
def replace_linear_layers(module, backend="triton_int4", quant_block_size=128):
    for name, child in module.named_children():
        if isinstance(child, nn.Linear) and name != "lm_head":
            quant_layer = QuantLinear.from_linear(
                child,
                backend=backend,
                quant_block_size=quant_block_size
            )
            setattr(module, name, quant_layer)
        else:
            replace_linear_layers(child, backend, quant_block_size)

In [ ]:
model_id = "unsloth/Llama-3.2-1B-Instruct" 
CONTEXT_LENGTH = 512 # Размер промпта из датасета
NUM_TOKENS = 128     # Сколько генерируем
    
print("Инициализация токенизатора и модели")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
    
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype=torch.float16, 
    device_map="cuda"
)
    
inputs = get_wikitext_inputs(tokenizer, context_length=CONTEXT_LENGTH)

In [ ]:
print("Замер Baseline (FP16)")
base_memory_static = get_allocated_memory_mb()
    
reset_memory_stats()
base_speed = measure_generation_speed(model, inputs, num_tokens=NUM_TOKENS)
base_memory_peak = get_peak_memory_mb()

In [ ]:
print("Квантование модели в INT4")
replace_linear_layers(model, backend="triton_int4", quant_block_size=128)

In [ ]:
print("Замер Quantized (INT4)")
quant_memory_static = get_allocated_memory_mb()
    
reset_memory_stats()
quant_speed = measure_generation_speed(model, inputs, num_tokens=NUM_TOKENS)
quant_memory_peak = get_peak_memory_mb()

In [ ]:
print(f"{'МЕТРИКА (Batch=1)':<25} | {'FP16 (Base)':<10} | {'INT4 (Triton)':<10}")
print(f"{'Память (Веса модели)':<25} | {base_memory_static:>7.0f} MB | {quant_memory_static:>7.0f} MB")
print(f"{'Память (Пик при ген.)':<25} | {base_memory_peak:>7.0f} MB | {quant_memory_peak:>7.0f} MB")
print(f"{'Скорость генерации':<25} | {base_speed:>7.1f} t/s | {quant_speed:>7.1f} t/s")